# COIL-100 objects: rotation next to the embedding

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# settings
DATA_DIR = "../data/coil-100"
HEAD = "embeddings"   # or "projections"
FRAME_MS = 100
PIXELS = "../data/coil_pixels.pt"   # dev/dump_pixels.py
RUNS = {
    "LocaLeReg": "../data/coil_embeddings_final_local_ballreg.pt",
    "SIGReg": "../data/coil_embeddings_final_sigreg.pt",
}

In [ ]:
import base64
import io
import sys
import uuid

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, Markdown, display
from IPython.display import Image as IPImage
from PIL import Image

sys.path.insert(0, "..")
from src.data.coil100_datamodule import COIL100Dataset
from visualisations import coil_topology_viz as viz

dataset = COIL100Dataset(DATA_DIR, lambda im: im, split="all")
runs = {name: viz.load_run(name, dump, dataset, HEAD) for name, dump in RUNS.items()}

In [ ]:
def scrubber(runs, obj):
    """A slider over one object's turntable frames: the frame itself next to its point in every run's PC1/PC2 plane.

    Self-contained HTML with the frames embedded, so it needs no kernel and survives an exported notebook.
    """
    runs = {runs.name: runs} if isinstance(runs, viz.Run) else runs
    planes = {}
    for name, run in runs.items():
        idx, plane, degrees = run.frames(obj)
        order = np.argsort(degrees)
        planes[name] = (idx[order], plane[order], degrees[order])
    paths, degrees = [dataset.paths[i] for i in next(iter(planes.values()))[0]], next(iter(planes.values()))[2]

    shots = []
    for k in range(len(degrees)):
        fig, axes = plt.subplots(1, 1 + len(runs), figsize=(3.5 * (1 + len(runs)), 3.6), squeeze=False)
        axes[0][0].imshow(Image.open(paths[k]))
        axes[0][0].axis("off")
        axes[0][0].set_title(f"{degrees[k]:.0f}°")
        for ax, (name, (_, plane, deg)) in zip(axes[0][1:], planes.items()):
            ax.scatter(plane[:, 0], plane[:, 1], c=deg, cmap="hsv", s=18, alpha=0.5)
            ax.scatter(plane[k, 0], plane[k, 1], s=180, facecolors="none", edgecolors="black", linewidths=2)
            ax.set_aspect("equal", adjustable="datalim")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{name}: PC1 / PC2")
        buffer = io.BytesIO()
        fig.tight_layout()
        fig.savefig(buffer, format="png", dpi=72)
        plt.close(fig)
        buffer.seek(0)
        palette = io.BytesIO()  # the notebook carries every frame, so palette PNGs instead of truecolor
        Image.open(buffer).convert("P", palette=Image.ADAPTIVE).save(palette, format="png", optimize=True)
        shots.append(palette.getvalue())

    tag = uuid.uuid4().hex[:8]
    images = "".join(
        f'<img src="data:image/png;base64,{base64.b64encode(shot).decode()}" style="display:{"block" if k == 0 else "none"};max-width:100%">'
        for k, shot in enumerate(shots)
    )
    labels = ", ".join(str(int(round(d))) for d in degrees)
    html = f"""<div id="{tag}">{images}
<input type="range" min="0" max="{len(shots) - 1}" value="0" style="width:100%"
 oninput="var d=document.getElementById('{tag}').getElementsByTagName('img');
          for (var i=0;i<d.length;i++) d[i].style.display = (i==this.value) ? 'block' : 'none';
          document.getElementById('{tag}-angle').textContent = [{labels}][this.value] + '\u00b0';">
<div style="text-align:center;font-family:sans-serif">frame <span id="{tag}-angle">{degrees[0]:.0f}\u00b0</span></div></div>"""
    return html


def _png(fig, dpi=110):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi)
    plt.close(fig)
    return buffer.getvalue()


def _img(data, kind):
    return f'<img src="data:image/{kind};base64,{base64.b64encode(data).decode()}" style="width:49%">'


def show_objects(run, chosen, frame_ms=100):
    """Per object: persistence + PC plane, then the rotation gif."""
    for obj in chosen:
        display(Markdown(f"### {run.name} · object {obj} · longest H1 {run.lifetimes[obj]:.3f}"))
        display(IPImage(data=_png(viz.object_figure(run, obj))))
        display(IPImage(data=viz.rotation_gif(dataset, run, obj, frame_ms)))


def show_pairs(runs, chosen, k=5, callouts=None, frame_ms=100):
    """Per object, one figure per run side by side: persistence + PC plane, rotation gif, PC plane with frame call-outs."""
    for obj in chosen:
        rows = {"png": [], "gif": [], "callout": []}
        at = callouts.get(obj) if isinstance(callouts, dict) else callouts
        for run in runs.values():
            rows["png"].append(_img(_png(viz.object_figure(run, obj)), "png"))
            rows["gif"].append(_img(viz.rotation_gif(dataset, run, obj, frame_ms), "gif"))
            figure = viz.callout_figure(dataset, run, obj, k, at)
            rows["callout"].append(_img(_png(figure), "png"))
        display(Markdown(f"### object {obj} · " + " vs ".join(f"{run.name} · longest H1 {run.lifetimes[obj]:.3f}" for run in runs.values())))
        display(HTML("".join(f'<div style="display:flex;gap:1%">{"".join(images)}</div>' for images in rows.values())))


## LocaLeReg

In [ ]:
show_objects(runs["LocaLeReg"], [2, 21, 28, 51, 67, 73, 74, 75, 84, 90], FRAME_MS) #1, 6, 21, 44, 51, 69. 76, 84, 96, 98

## LocaLeReg: its own top-10 by persistence

In [ ]:
show_objects(runs["LocaLeReg"], [1, 6, 21, 44, 51, 69, 76, 84, 96, 98], FRAME_MS)

## SIGReg

In [ ]:
show_objects(runs["SIGReg"], [2, 21, 28, 51, 67, 73, 74, 75, 84, 90], FRAME_MS)

## Grid: LocaLeReg vs SIGReg

In [ ]:
objs_local = [44, 6, 76, 1, 35, 22, 65, 84]
objs_sigreg = [7, 2, 12]

### LocaLeReg > SIGReg

In [ ]:
show_pairs(runs, objs_local, frame_ms=FRAME_MS)

### SIGReg > LocaLeReg

In [ ]:
show_pairs(runs, objs_sigreg, frame_ms=FRAME_MS)

## Grid: one row per object

In [ ]:
viz.grid_figure(dataset, runs, [1, 22, 35, 76], columns=viz.GRID_COLUMNS, callouts={
    1: (0, 70, 145, 215, 290),
    22: (0, 60, 110, 225, 310),
    35: (25, 80, 170, 210, 310),
    76: (75, 90, 120, 210, 345)
    });

## Scrubber: one object frame by frame

In [ ]:
HTML(scrubber(runs, 76))

## Overlay: several objects in one shared plane

In [ ]:
panels = {"pixels": viz.load_run("pixels", PIXELS, dataset, HEAD), **runs}


def overlay(chosen, title):
    fig, rows = viz.overlay_figure(panels, chosen, title)
    print(viz.geometry_table(rows))

In [ ]:
overlay(viz.top(panels["pixels"]), "top-5 by pixel persistence")

In [ ]:
overlay(viz.top(panels["LocaLeReg"]), "top-5 by LocaLeReg persistence")

In [ ]:
overlay(viz.top(panels["SIGReg"]), "top-5 by SIGReg persistence")

Each object's 72 frames are a closed curve in any space: frame 0° = frame 360° and neighbours are close, so ripser finds an $H_1$ bar in pixels already (median 0.39 vs 0.19 in either latent). PC1/PC2 of one object is the plane of its dominant angular harmonic; in pixels that is usually $n=2$ (silhouette has period $\pi$).